# Person A: baseline model (embedding + fully connected network)

You are building the **baseline** that the other two models are compared against.

**How this notebook works:** the data, splits, training loop, early stopping and result logging are shared (`common.py`, do not edit it). **You write the model** in the cell marked `YOUR MODEL`; every other cell already works. Runtime > Change runtime type > GPU (T4), then run cells top to bottom.

In [ ]:
%pip install -q datasets transformers
import os
if not os.path.exists("common.py"):
    REPO_URL = "https://github.com/ZhongVang/csci167-rotten-tomatoes.git"   # group repo (private until submission)
    !git clone $REPO_URL project
    %cd project

In [ ]:
# Optional (Colab): keep results on Drive so they survive session timeouts.
# Must run BEFORE `import common`.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ["RESULTS_DIR"] = "/content/drive/MyDrive/CSCI167_results"
except Exception:
    pass  # running locally: results go to ./results

In [ ]:
OWNER = "A"
RESULTS_FILE = f"results_{OWNER}.csv"
SEEDS = [0, 1, 2]                    # shared: do not change

import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from common import *

print(save_env_info(OWNER))          # package versions + hardware, for the report
data = load_data()
print({k: len(v[0]) for k, v in data.items()}, "-> train/val/test sizes")
print("label counts (train):", np.bincount(data["train"][1]))
print("example:", data["train"][0][0], "->", data["train"][1][0])

## YOUR MODEL

**Suggested architecture** (textbook Ch 3-4, fully connected networks):
1. `nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)` turns each token into a vector.
2. **Pool over the sentence** to get one vector per sentence, ignoring padding. A masked mean is the simplest:
   `(emb * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)`. This throws away word order on purpose.
3. One or more `nn.Linear` + `nn.ReLU` (+ `nn.Dropout`) layers, then a final `nn.Linear(..., 2)`.

**Things worth trying and explaining in the report:** how many hidden layers (0, 1, 2)? embedding size? dropout?
Because this is the baseline, keep it simple. Add a model hyperparameter as an `__init__` argument
(e.g. `dropout=0.5`) so you can vary it with `run(seed, frac, dropout=0.0)`.

In [ ]:
class MyModel(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden=128, dropout=0.5):
        super().__init__()
        # TODO: define your layers here.
        # vocab_size = number of tokens in the vocabulary (index 0 is the padding token)
        raise NotImplementedError

    def forward(self, ids, mask):
        # ids:  LongTensor [batch, length]  token indices (0 = padding)
        # mask: BoolTensor [batch, length]  True where the token is real, False where it is padding
        # return: logits, FloatTensor [batch, 2]   (class 0 = negative, 1 = positive)
        raise NotImplementedError

## Shared training settings (do not change for the main runs)

Every model in the group is trained with the same budget so the comparison is fair: Adam, lr 1e-3, batch 64, at most 20 epochs, early stopping (patience 5) on **validation** accuracy; the best-validation weights are restored and the test set is evaluated once. The same 3 seeds are used for 100%, 25% and 10% of the training data.

In [ ]:
TAG = "mlp"                 # short unique name used in run names / files
MODEL_NAME = "MeanPoolMLP"   # TODO: rename to describe YOUR model (shown in the results table)

PROTOCOL = dict(optimizer="adam", lr=1e-3, batch_size=64, epochs=20, patience=5)
FRACS = [1.0, 0.25, 0.1]

def run(seed, frac, tag=None, **overrides):
    """Train + evaluate one run and log it to results/{RESULTS_FILE}.
    overrides: training settings (optimizer, lr, weight_decay, schedule, momentum) go to the trainer;
    anything else is passed to MyModel(...) as a model hyperparameter (e.g. dropout=0.0)."""
    train_keys = {"optimizer", "lr", "weight_decay", "schedule", "momentum"}
    tcfg = {**PROTOCOL, **{k: v for k, v in overrides.items() if k in train_keys}}
    mcfg = {k: v for k, v in overrides.items() if k not in train_keys}
    set_seed(seed)                                    # BEFORE building the model
    train_dl, val_dl, test_dl, vocab = make_vocab_loaders(data, tcfg["batch_size"], seed, train_frac=frac)
    model = MyModel(len(vocab), **mcfg)
    return train_and_eval(model, (train_dl, val_dl, test_dl), run_name=f"{tag or TAG}_f{frac}_s{seed}",
                          model_name=MODEL_NAME, seed=seed, train_frac=frac, extra=mcfg,
                          results_file=RESULTS_FILE, **tcfg)

## Sanity check (run this before training)

Builds your model, prints its structure and trainable parameter count (needed for the report), and checks the output shape on a real batch.

In [ ]:
set_seed(0)
train_dl, val_dl, test_dl, vocab = make_vocab_loaders(data, 64, 0)
model = MyModel(len(vocab))
print(model)
print("trainable / total params:", count_params(model))
inputs, y = next(iter(train_dl))
out = model(**inputs)
assert out.shape == (len(y), 2), f"forward() must return logits of shape [batch, 2], got {tuple(out.shape)}"
print("OK: logits", tuple(out.shape), "| ids", tuple(inputs['ids'].shape))

Quick test (one short run, 10% of the data). If this trains and the validation accuracy is above 50%, launch the full runs below.

In [ ]:
run(0, 0.1, tag=TAG + '_test')

## Main runs: 3 seeds x 100% / 25% / 10% of the training data

In [ ]:
for frac in FRACS:
    for seed in SEEDS:
        run(seed, frac)

## Results: summary table, learning curves, error analysis

In [ ]:
summarize(f'results_{OWNER}.csv')   # mean +/- std over seeds, per configuration

In [ ]:
plot_history(f"{TAG}_f1.0_s0", f"{MODEL_NAME}: train vs validation (seed 0)");

In [ ]:
import json
h = json.load(open(f"{RESULTS_DIR}/history/{TAG}_f1.0_s0.json"))
preds, labels = np.array(h["test_preds"]), np.array(data["test"][1])
texts = data["test"][0]
cm = np.zeros((2, 2), int)
for y, p in zip(labels, preds): cm[y, p] += 1
print("confusion matrix (rows=true neg/pos, cols=pred neg/pos):\n", cm)
wrong = np.where(preds != labels)[0]
print(len(wrong), "errors. Examples (true, pred, text):")
for i in wrong[:20]: print(labels[i], preds[i], "|", texts[i])

## Notes for the report (fill in)

- **Design choices:** why this architecture, why these sizes/layers? What did you expect it to do well or badly?
- **Learning curves:** does it overfit (train accuracy far above validation)? When does validation peak?
- **Errors:** what do the misclassified sentences have in common (negation, sarcasm, mixed opinions, very short text)?
- **Less data:** how much does accuracy drop at 25% and 10%?
- **Limitations:** 3 seeds, small test set (about 1,066 sentences), anything you would do differently.